# Ceftriaxone — Species-Masked Federated Learning

**Federated species masking per site, then compare 3 mask strategies + unmasked.**

| Phase | Content |
|---|---|
| 1 | Per-site RF trained to classify species → saved for 08 reuse |
| 2 | 3 mask strategies (union, majority, persite) + unmasked baseline |
| 3 | Mask comparison: centralized MLP + FedAvg MLP for each mask |
| 4 | Best mask → full FL pipeline (FedProx μ=0.1, FedLR, FedRF) |
| Output | RF models + masks saved for 08 pipeline reuse |

All species (no E. coli filter). Species-stratified splits.

In [ ]:
# ── CONFIG ──
MASK_TOP_K = 500
NUM_ROUNDS = 30
NUM_RF_ROUNDS = 5
RUN_NAME = "06-03c-Ceftriaxone-Federated-z-Runs"
TARGET_RUN = "01-Run"  # subdirectory within RUN_NAME

In [ ]:
!pip install "flwr[simulation]" maldideepkit maldiamrkit seaborn --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
import warnings, os, io, shutil, json as _json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     cross_val_predict, StratifiedKFold)
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score)

from maldideepkit.attention.mlp import SpectralAttentionMLP
from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldiamrkit.evaluation import stratified_species_drug_split

import flwr as fl
import joblib

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
FL_DEVICE = "cpu"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Flower: {fl.__version__}  |  Device: {DEV}")

In [ ]:
if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
else:
    DRYAD = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet")

DRUG_NAME = "Ceftriaxone"

PROJECT_DIR = DRYAD / "Processed/Processing/Analysis/06-Ceftazidime-E-coli" / RUN_NAME
RUN_DIR = PROJECT_DIR / TARGET_RUN
OUT_DIR = RUN_DIR / "results"; MODEL_DIR = RUN_DIR / "models"
MASK_DIR = RUN_DIR / "masks"; RF_DIR = RUN_DIR / "rf_species_models"
for d in [OUT_DIR, MODEL_DIR, MASK_DIR, RF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SITES_PATHS = {
    "A": DRYAD / "Processed/Proc_DRIAMS-A" / DRUG_NAME / "data.csv",
    "B": DRYAD / "Processed/Proc_DRIAMS-B" / DRUG_NAME / "data.csv",
    "C": DRYAD / "Processed/Proc_DRIAMS-C" / DRUG_NAME / "data.csv",
    "D": DRYAD / "Processed/Proc_DRIAMS-D" / DRUG_NAME / "data.csv",
}
SITE_ORDER = ["A", "B", "C", "D"]
print(f"Drug: {DRUG_NAME}  |  Run: {RUN_DIR}")

In [ ]:
# ── Shared constants ──
THRESHOLDS = np.linspace(0.05, 0.95, 91)
LR_GRID = np.linspace(1e-4, 5e-4, 6)
DROP_GRID = np.linspace(0.2, 0.6, 6)
LOCAL_EPOCHS = 1; BATCH_SIZE = 16
FEDPROX_MUS = [0.1]
MASK_STRATEGIES = ["none", "union", "majority", "persite"]

RF_PARAM_GRID = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [10, 20, 30, None],
    "min_samples_leaf": [2, 5, 10],
    "class_weight": ["balanced", "balanced_subsample"],
}
C_GRID = np.linspace(5e-5, 1e-3, 15)

def n_mixes(n_train):
    if n_train < 500: return 5
    if n_train < 1500: return 4
    if n_train < 5000: return 3
    return 1

In [ ]:
# ── Load Ceftriaxone — ALL species ──
raw_data = {}
for site, path in SITES_PATHS.items():
    df = pd.read_csv(path)
    bin_cols = [c for c in df.columns if c.startswith("bin_")]
    X = df[bin_cols].to_numpy(dtype="float32")
    y = df["label"].to_numpy(dtype="int64")
    species = df["species"].values
    raw_data[site] = (X, y, species)
    nr, ns = (y==1).sum(), (y==0).sum()
    n_sp = len(np.unique(species))
    print(f"  Site {site}: {len(y)} samples ({ns} S, {nr} R, {nr/len(y)*100:.1f}% R, {n_sp} species)")
print(f"Total: {sum(len(raw_data[s][1]) for s in SITE_ORDER)}")

In [ ]:
# ── Per-site species-stratified 90/10 split ──
client_train = {}; client_test = {}
species_train = {}; species_test = {}

for site in SITE_ORDER:
    X, y, sp = raw_data[site]
    n = len(y); idx = np.arange(n).reshape(-1,1)
    itr, iv, _, _ = stratified_species_drug_split(idx, y, species=sp, test_size=0.10, random_state=SEED)
    itr = itr.flatten().astype(int); iv = iv.flatten().astype(int)
    client_train[site] = (X[itr], y[itr]); client_test[site] = (X[iv], y[iv])
    species_train[site] = sp[itr]; species_test[site] = sp[iv]
    print(f"  Site {site}: train={len(itr)} test={len(iv)} species={len(np.unique(sp[itr]))}")

pooled_X_train = np.concatenate([client_train[s][0] for s in SITE_ORDER])
pooled_y_train = np.concatenate([client_train[s][1] for s in SITE_ORDER])
pooled_sp_train = np.concatenate([species_train[s] for s in SITE_ORDER])
print(f"\nPooled train: {len(pooled_X_train)} samples, {len(np.unique(pooled_sp_train))} species")

In [ ]:
# ── Per-site preprocessing ──
client_train_pp = {}; client_test_pp = {}

for site in SITE_ORDER:
    X_tr, y_tr = client_train[site]
    X_te, y_te = client_test[site]
    state = fit_input_transform(X_tr, "log1p+standardize")
    client_train_pp[site] = (apply_input_transform(X_tr, state), y_tr)
    client_test_pp[site] = (apply_input_transform(X_te, state), y_te)
print("Per-site preprocessing done. Data still unmasked at this point.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PHASE 1: Per-site RF species models + save
# ═══════════════════════════════════════════════════════════════════════════

print("\n=== Phase 1: Per-Site RF Species Models ===")
site_importances = {}
site_masks_raw = {}   # top-K indices per site
site_n_species = {}

for site in SITE_ORDER:
    X_tr, y_tr = client_train_pp[site]
    sp_tr = species_train[site]
    n_sp = len(np.unique(sp_tr)); site_n_species[site] = n_sp

    if n_sp < 3:
        print(f"  Site {site}: only {n_sp} species — species masking would be meaningless")
        site_importances[site] = np.ones(6000) / 6000  # uniform
        site_masks_raw[site] = np.array([], dtype=int)
        continue

    rf_sp = RandomForestClassifier(n_estimators=300, max_depth=20, min_samples_leaf=5,
                                    n_jobs=-1, oob_score=True, random_state=SEED)
    rf_sp.fit(X_tr, sp_tr)
    # Save for 08 reuse
    joblib.dump(rf_sp, RF_DIR / f"site_{site}_rf_species.joblib")

    site_importances[site] = rf_sp.feature_importances_
    top_k = np.argsort(rf_sp.feature_importances_)[-MASK_TOP_K:]
    top_k.sort()
    site_masks_raw[site] = top_k
    explained = rf_sp.feature_importances_[top_k].sum()
    print(f"  Site {site}: {n_sp} species, OOB acc={rf_sp.oob_score_:.4f}, "
          f"top-{MASK_TOP_K} bins explain {explained:.1%} of species importance")

print("\nPhase 1 done. RF models saved to rf_species_models/.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PHASE 2: Compute mask strategies + create masked data copies
# ═══════════════════════════════════════════════════════════════════════════

print("\n=== Phase 2: Mask Computation ===")

# ── Union mask: any bin flagged by any site ──
all_site_bins = [site_masks_raw[s] for s in SITE_ORDER if len(site_masks_raw[s]) > 0]
union_mask = np.unique(np.concatenate(all_site_bins)) if all_site_bins else np.array([], dtype=int)
print(f"  Union mask: {len(union_mask)} bins")

# ── Majority mask: bin flagged by >= 2 sites ──
bin_counts = {}
for bins in all_site_bins:
    for b in bins:
        bin_counts[b] = bin_counts.get(b, 0) + 1
majority_mask = np.array([b for b, c in bin_counts.items() if c >= 2], dtype=int)
majority_mask.sort()
print(f"  Majority mask: {len(majority_mask)} bins (flagged by >=2 sites)")

# ── Persite mask: each site uses its own top-K (already stored in site_masks_raw) ──
print(f"  Persite mask: {MASK_TOP_K} bins per site")

# Save all masks
np.save(MASK_DIR / "union_mask.npy", union_mask)
np.save(MASK_DIR / "majority_mask.npy", majority_mask)
for site in SITE_ORDER:
    np.save(MASK_DIR / f"persite_site_{site}_mask.npy", site_masks_raw[site])

# ── Overlap between sites ──
print("\n  Pairwise mask overlaps (% shared bins):")
sites_with_mask = [s for s in SITE_ORDER if len(site_masks_raw[s]) > 0]
for i, sa in enumerate(sites_with_mask):
    for j, sb in enumerate(sites_with_mask):
        if j <= i: continue
        overlap = len(set(site_masks_raw[sa]) & set(site_masks_raw[sb]))
        pct = overlap / MASK_TOP_K * 100
        print(f"    {sa}∩{sb}: {overlap}/{MASK_TOP_K} ({pct:.0f}%)")

In [ ]:
# ── Create masked data variants for each strategy ──
# We keep the unmasked data in client_train_pp_unmasked for comparison

client_train_unmasked = {s: (client_train_pp[s][0].copy(), client_train_pp[s][1].copy())
                          for s in SITE_ORDER}
client_test_unmasked  = {s: (client_test_pp[s][0].copy(), client_test_pp[s][1].copy())
                          for s in SITE_ORDER}

# pooled preprocess for centralized
state_pool = fit_input_transform(pooled_X_train, "log1p+standardize")
X_pool_unmasked = apply_input_transform(pooled_X_train, state_pool).copy()

# Build test sets from preprocessed client_test
test_dict_unmasked = {s: (client_test_unmasked[s][0], client_test_unmasked[s][1]) for s in SITE_ORDER}
combined_X_test_unmasked = np.concatenate([test_dict_unmasked[s][0] for s in SITE_ORDER])
combined_y_test = np.concatenate([test_dict_unmasked[s][1] for s in SITE_ORDER])

# Masked variants
masked_data = {}  # {strategy: {train: {site: (X,y)}, test: {site: (X,y)}, pooled_X, pooled_y}}

for strategy in MASK_STRATEGIES:
    if strategy == "none":
        # Use unmasked data directly
        masked_data[strategy] = {"train": client_train_unmasked, "test": client_test_unmasked,
                                  "pooled_X": X_pool_unmasked, "pooled_y": pooled_y_train}
        continue

    # Determine the mask for this strategy
    if strategy == "union":
        global_mask = union_mask
    elif strategy == "majority":
        global_mask = majority_mask

    tr = {}; te = {}
    for site in SITE_ORDER:
        if strategy == "persite":
            mask = site_masks_raw[site] if len(site_masks_raw[site]) > 0 else np.array([], dtype=int)
        else:
            mask = global_mask
        if len(mask) == 0:
            tr[site] = (client_train_unmasked[site][0].copy(), client_train_unmasked[site][1].copy())
            te[site] = (client_test_unmasked[site][0].copy(), client_test_unmasked[site][1].copy())
        else:
            X_tr = client_train_unmasked[site][0].copy(); X_tr[:, mask] = 0.0
            X_te = client_test_unmasked[site][0].copy(); X_te[:, mask] = 0.0
            tr[site] = (X_tr, client_train_unmasked[site][1].copy())
            te[site] = (X_te, client_test_unmasked[site][1].copy())

    X_pool = X_pool_unmasked.copy()
    if len(global_mask if strategy != "persite" else np.array([])) > 0 and strategy != "persite":
        X_pool[:, global_mask] = 0.0

    masked_data[strategy] = {"train": tr, "test": te, "pooled_X": X_pool, "pooled_y": pooled_y_train}

print("Masked data variants created for all strategies.")

In [ ]:
# ── Model construction + helpers ──

class DatasetFromNumpy(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32); self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

def build_mlp(dropout_high):
    return SpectralAttentionMLP(
        input_dim=6000, n_classes=2, hidden_dim=512, head_dims=(256,128),
        dropout_high=dropout_high, dropout_low=dropout_high/2.0, use_attention=False)

def model_to_numpy(model):
    return [v.cpu().numpy() for v in model.state_dict().values()]

def numpy_to_model(model, params):
    sd = model.state_dict()
    for k, p in zip(sd.keys(), params): sd[k] = torch.tensor(p)
    model.load_state_dict(sd)

def mlp_predict_proba(model, X_np, dev=DEV):
    model.eval()
    X_t = torch.tensor(X_np, dtype=torch.float32).to(dev)
    with torch.no_grad():
        return F.softmax(model(X_t), dim=1).cpu().numpy()[:, 1]

In [ ]:
# ── Centralized MLP grid search (unmasked, pooled data) ──
print("\n=== Centralized MLP Grid Search (unmasked) ===")
X_gs, X_gv, y_gs, y_gv = train_test_split(
    X_pool_unmasked, pooled_y_train, test_size=0.15, stratify=pooled_y_train, random_state=SEED)

best_ba, BEST_MLP_LR, BEST_MLP_DH, BEST_MLP_THRESH = -1.0, None, None, 0.5
for lr in LR_GRID:
    for d in DROP_GRID:
        m = build_mlp(d).to(DEV)
        ds = DatasetFromNumpy(X_gs, y_gs)
        dl = DataLoader(ds, batch_size=64, shuffle=True)
        opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-3)
        crit = nn.CrossEntropyLoss()
        for _ in range(50):
            m.train()
            for xb, yb in dl:
                if len(xb) <= 1: continue
                xb, yb = xb.to(DEV), yb.to(DEV)
                opt.zero_grad(); crit(m(xb), yb).backward(); opt.step()
        proba = mlp_predict_proba(m, X_gv)
        for t in THRESHOLDS:
            ba = balanced_accuracy_score(y_gv, proba >= t)
            if ba > best_ba: best_ba = ba; BEST_MLP_LR = lr; BEST_MLP_DH = d; BEST_MLP_THRESH = t
    print(f"  lr={lr:.1e}  best-drop={BEST_MLP_DH:.1f}  BA={best_ba:.4f}")
print(f"\nBest: lr={BEST_MLP_LR:.1e} drop={BEST_MLP_DH:.1f} thresh={BEST_MLP_THRESH:.3f}")

In [ ]:
# ── Train centralized MLP + RF on unmasked pooled ──
print("\n=== Centralized MLP (unmasked) ===")
cent_m = build_mlp(BEST_MLP_DH).to(DEV)
ds = DatasetFromNumpy(X_pool_unmasked, pooled_y_train)
dl = DataLoader(ds, batch_size=64, shuffle=True)
opt = torch.optim.AdamW(cent_m.parameters(), lr=BEST_MLP_LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=90, eta_min=1e-6)
crit = nn.CrossEntropyLoss(); best_loss, best_sd, patience = float("inf"), None, 0
for ep in range(100):
    cent_m.train()
    for xb, yb in dl:
        if len(xb) <= 1: continue
        xb, yb = xb.to(DEV), yb.to(DEV)
        if ep < 10:
            for pg in opt.param_groups: pg["lr"] = BEST_MLP_LR * (ep+1)/10
        opt.zero_grad(); crit(cent_m(xb), yb).backward(); opt.step()
    if ep >= 10: sched.step()
    if ep % 5 == 0:
        cent_m.eval()
        with torch.no_grad():
            vl = sum(crit(cent_m(xb.to(DEV)), yb.to(DEV)).item() for xb, yb in dl)/len(dl)
        if vl < best_loss: best_loss = vl; best_sd = {k:v.cpu().clone() for k,v in cent_m.state_dict().items()}; patience = 0
        else: patience += 1
        if patience >= 3: break
if best_sd: cent_m.load_state_dict(best_sd)
cent_m.eval()

centralized_mlp_results = {}
for site in SITE_ORDER:
    X_tt = client_test_unmasked[site][0]; y_tt = client_test_unmasked[site][1]
    proba = mlp_predict_proba(cent_m, X_tt)
    centralized_mlp_results[f"{site}_BalAcc"] = balanced_accuracy_score(y_tt, proba >= BEST_MLP_THRESH)
    centralized_mlp_results[f"{site}_AUC"] = roc_auc_score(y_tt, proba)
proba_all = mlp_predict_proba(cent_m, combined_X_test_unmasked)
centralized_mlp_results["All_BalAcc"] = balanced_accuracy_score(combined_y_test, proba_all >= BEST_MLP_THRESH)
centralized_mlp_results["All_AUC"] = roc_auc_score(combined_y_test, proba_all)
print(f"Centralized MLP (unmasked): All_BalAcc={centralized_mlp_results['All_BalAcc']:.4f}")

# Centralized RF
grid_rf = GridSearchCV(RandomForestClassifier(oob_score=True, random_state=SEED, n_jobs=-1),
                        param_grid=RF_PARAM_GRID, cv=3, scoring="balanced_accuracy", n_jobs=-1)
grid_rf.fit(X_pool_unmasked, pooled_y_train); RF_PARAMS = grid_rf.best_params_
print(f"RF best: {RF_PARAMS}")
rf_cent = grid_rf.best_estimator_

centralized_rf_results = {}
for site in SITE_ORDER:
    X_tt = client_test_unmasked[site][0]; y_tt = client_test_unmasked[site][1]
    proba = rf_cent.predict_proba(X_tt)[:,1]
    centralized_rf_results[f"{site}_BalAcc"] = balanced_accuracy_score(y_tt, proba >= 0.5)
    centralized_rf_results[f"{site}_AUC"] = roc_auc_score(y_tt, proba)
proba_all = rf_cent.predict_proba(combined_X_test_unmasked)[:,1]
centralized_rf_results["All_BalAcc"] = balanced_accuracy_score(combined_y_test, proba_all >= 0.5)
centralized_rf_results["All_AUC"] = roc_auc_score(combined_y_test, proba_all)
print(f"Centralized RF (unmasked): All_BalAcc={centralized_rf_results['All_BalAcc']:.4f}")
BEST_RF_THRESH = 0.5; RF_TREES_PER_ROUND = RF_PARAMS["n_estimators"]
print(f"BEST_MLP_LR={BEST_MLP_LR:.2e} BEST_MLP_DH={BEST_MLP_DH:.1f} RF_TREES={RF_TREES_PER_ROUND}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PHASE 3: Mask comparison — Centralized MLP per mask
# ═══════════════════════════════════════════════════════════════════════════

print("\n=== Phase 3: Mask Comparison (Centralized MLP) ===")
mask_comparison = {"none": centralized_mlp_results}

for strategy in ["union", "majority", "persite"]:
    print(f"\n--- {strategy} mask ---")
    data = masked_data[strategy]
    X_pool = data["pooled_X"]

    m = build_mlp(BEST_MLP_DH).to(DEV)
    ds = DatasetFromNumpy(X_pool, data["pooled_y"])
    dl = DataLoader(ds, batch_size=64, shuffle=True)
    opt = torch.optim.AdamW(m.parameters(), lr=BEST_MLP_LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=90, eta_min=1e-6)
    crit = nn.CrossEntropyLoss()
    best_loss, best_sd, patience = float("inf"), None, 0
    for ep in range(100):
        m.train()
        for xb, yb in dl:
            if len(xb) <= 1: continue
            xb, yb = xb.to(DEV), yb.to(DEV)
            if ep < 10:
                for pg in opt.param_groups: pg["lr"] = BEST_MLP_LR * (ep+1)/10
            opt.zero_grad(); crit(m(xb), yb).backward(); opt.step()
        if ep >= 10: sched.step()
        if ep % 5 == 0:
            m.eval()
            with torch.no_grad():
                vl = sum(crit(m(xb.to(DEV)), yb.to(DEV)).item() for xb, yb in dl)/len(dl)
            if vl < best_loss: best_loss = vl; best_sd = {k:v.cpu().clone() for k,v in m.state_dict().items()}; patience = 0
            else: patience += 1
            if patience >= 3: break
    if best_sd: m.load_state_dict(best_sd)
    m.eval()

    results = {}
    for site in SITE_ORDER:
        X_tt = data["test"][site][0]; y_tt = data["test"][site][1]
        proba = mlp_predict_proba(m, X_tt)
        results[f"{site}_BalAcc"] = balanced_accuracy_score(y_tt, proba >= BEST_MLP_THRESH)
        results[f"{site}_AUC"] = roc_auc_score(y_tt, proba)
    combined_X = np.concatenate([data["test"][s][0] for s in SITE_ORDER])
    combined_y = np.concatenate([data["test"][s][1] for s in SITE_ORDER])
    proba_all = mlp_predict_proba(m, combined_X)
    results["All_BalAcc"] = balanced_accuracy_score(combined_y, proba_all >= BEST_MLP_THRESH)
    results["All_AUC"] = roc_auc_score(combined_y, proba_all)
    for site in SITE_ORDER:
        delta = results[f"{site}_BalAcc"] - centralized_mlp_results[f"{site}_BalAcc"]
        print(f"  {site}: masked={results[f'{site}_BalAcc']:.4f}  unmasked={centralized_mlp_results[f'{site}_BalAcc']:.4f}  Δ={delta:+.4f}")
    check_delta = results["All_BalAcc"] - centralized_mlp_results["All_BalAcc"]
    print(f"  All: masked={results['All_BalAcc']:.4f}  unmasked={centralized_mlp_results['All_BalAcc']:.4f}  Δ={check_delta:+.4f}")
    mask_comparison[strategy] = results

# Print summary table
print("\n  Mask Comparison Summary (Centralized MLP All_BalAcc):")
for s in MASK_STRATEGIES:
    print(f"    {s:10s}: {mask_comparison[s].get('All_BalAcc', np.nan):.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PHASE 3 continued: FedAvg MLP per mask
# ═══════════════════════════════════════════════════════════════════════════

# ── FL infra (train_local_mixed, FedMLPClient, etc.) ──
def train_local_mixed(model, X_np, y_np, n_mixes_val, proximal_mu, global_params):
    collect_sds = []; total_loss = 0.0
    for mix_i in range(n_mixes_val):
        seed = SEED + mix_i * 100 + int(proximal_mu * 1000)
        torch.manual_seed(seed); np.random.seed(seed)
        idx = np.random.permutation(len(X_np))
        X_shuf, y_shuf = X_np[idx], y_np[idx]
        ds = DatasetFromNumpy(X_shuf, y_shuf)
        dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
        opt = torch.optim.AdamW(model.parameters(), lr=BEST_MLP_LR, weight_decay=1e-4)
        crit = nn.CrossEntropyLoss()
        model.train(); batch_loss = 0.0; n_batch = 0
        for xb, yb in dl:
            if len(xb) <= 1: continue
            xb, yb = xb.to(FL_DEVICE), yb.to(FL_DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            if proximal_mu > 0 and global_params is not None:
                prox = sum((w - gw.to(FL_DEVICE)).norm(2) for w, gw in zip(model.parameters(), global_params))
                loss = loss + (proximal_mu / 2.0) * prox
            loss.backward(); opt.step()
            batch_loss += loss.item(); n_batch += 1
        total_loss += batch_loss / n_batch
        collect_sds.append({k: v.cpu().clone() for k, v in model.state_dict().items()})
    if len(collect_sds) > 1:
        avg_sd = {}
        for k in collect_sds[0].keys():
            stacked = torch.stack([sd[k].float() for sd in collect_sds])
            avg_sd[k] = stacked.mean(0)
            if k.endswith("num_batches_tracked"):
                avg_sd[k] = avg_sd[k].to(torch.long) if avg_sd[k].dim()==0 else avg_sd[k].round().to(torch.long)
        model.load_state_dict(avg_sd)
    return total_loss / n_mixes_val

class FedMLPClient(fl.client.NumPyClient):
    def __init__(self, cid, X_train, y_train):
        self.cid = cid; self.X_train, self.y_train = X_train, y_train
        self.n_mixes_val = n_mixes(len(X_train))
        self.model = build_mlp(BEST_MLP_DH).to(FL_DEVICE)
    def get_parameters(self, config): return model_to_numpy(self.model)
    def set_parameters(self, params): numpy_to_model(self.model, params)
    def fit(self, parameters, config):
        self.set_parameters(parameters)
        proximal_mu = float(config.get("proximal_mu", 0.0))
        global_copy = None
        if proximal_mu > 0: global_copy = [p.clone().detach() for p in self.model.parameters()]
        loss = train_local_mixed(self.model, self.X_train, self.y_train,
                                 self.n_mixes_val, proximal_mu, global_copy)
        return (self.get_parameters({}), len(self.X_train),
                {"train_loss": loss, "n_mixes": self.n_mixes_val})

def get_mlp_eval_fn(test_dict, threshold, hist_list):
    def evaluate(server_round, parameters, config):
        m = build_mlp(BEST_MLP_DH); numpy_to_model(m, parameters); m.to(FL_DEVICE); m.eval()
        record = {"round": server_round}
        ap, al = [], []
        for site in SITE_ORDER:
            X_tt, y_tt = test_dict[site]
            proba = mlp_predict_proba(m, X_tt, dev=FL_DEVICE)
            preds = proba >= threshold
            record[f"{site}_BalAcc"] = float(balanced_accuracy_score(y_tt, preds))
            record[f"{site}_AUC"] = float(roc_auc_score(y_tt, proba))
            ap.append(proba); al.append(y_tt)
        apc = np.concatenate(ap); alc = np.concatenate(al)
        record["All_BalAcc"] = float(balanced_accuracy_score(alc, apc >= threshold))
        record["All_AUC"] = float(roc_auc_score(alc, apc))
        hist_list.append(record)
        return (1.0 - record["All_BalAcc"], record)
    return evaluate

print("\n=== Phase 3 continued: FedAvg MLP per mask ===")
fedavg_hist_per_mask = {}

for strategy in MASK_STRATEGIES:
    print(f"\n--- {strategy} mask ---")
    data = masked_data[strategy]
    fedavg_hist = []; _eval_fn = get_mlp_eval_fn(data["test"], BEST_MLP_THRESH, fedavg_hist)

    strategy_obj = fl.server.strategy.FedAvg(
        fraction_fit=1.0, fraction_evaluate=0.0,
        min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4,
        evaluate_fn=_eval_fn,
        initial_parameters=fl.common.ndarrays_to_parameters(model_to_numpy(build_mlp(BEST_MLP_DH))))

    def make_client_fn(train_dict):
        def _fn(cid):
            site = SITE_ORDER[int(cid)]
            return FedMLPClient(cid, *train_dict[site]).to_client()
        return _fn

    fl.simulation.start_simulation(
        client_fn=make_client_fn(data["train"]), num_clients=4,
        config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
        strategy=strategy_obj, client_resources={"num_cpus": 1, "num_gpus": 0})

    fedavg_hist_per_mask[strategy] = fedavg_hist.copy()
    last = fedavg_hist[-1] if fedavg_hist else {}
    print(f"  Final All_BalAcc: {last.get('All_BalAcc', np.nan):.4f}")
    print(f"  Best All_BalAcc:  {max((h.get('All_BalAcc',0) for h in fedavg_hist[1:]), default=0):.4f}")

In [ ]:
# ── Pick best mask: highest worst-site improvement ──
print("\n=== Best Mask Selection ===")
best_strategy = "none"; best_worst = 0
for strategy in MASK_STRATEGIES:
    delta_per_site = []
    for site in SITE_ORDER:
        if strategy in fedavg_hist_per_mask and fedavg_hist_per_mask[strategy]:
            masked_val = max((h.get(f"{site}_BalAcc",0) for h in fedavg_hist_per_mask[strategy][1:]), default=0)
            unmasked_val = max((h.get(f"{site}_BalAcc",0) for h in fedavg_hist_per_mask["none"][1:]), default=0)
            delta_per_site.append(masked_val - unmasked_val)
    if delta_per_site:
        worst = min(delta_per_site)
        print(f"  {strategy:10s}: worst-site Δ={worst:+.4f}")
        if worst > best_worst:
            best_worst = worst; best_strategy = strategy

print(f"\nBest mask: {best_strategy} (worst-site improvement: {best_worst:+.4f})")

In [ ]:
# ── Switch active data to best mask ──
client_train_pp = masked_data[best_strategy]["train"]
client_test_pp = masked_data[best_strategy]["test"]
X_best_pool = masked_data[best_strategy]["pooled_X"]
print(f"Switched to best mask: {best_strategy}")
print(f"  Masked bins: ", end="")
if best_strategy == "union": print(f"{len(union_mask)} (union)")
elif best_strategy == "majority": print(f"{len(majority_mask)} (majority)")
elif best_strategy == "persite": print(f"~{MASK_TOP_K} per site (persite)")
else: print("none (unmasked)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PHASE 4: Full FL pipeline with best mask
# ═══════════════════════════════════════════════════════════════════════════

In [ ]:
fedavg_hist = fedavg_hist_per_mask[best_strategy]
print(f"FedAvg MLP ({best_strategy} mask): {len(fedavg_hist)} rounds available from Phase 3")

In [ ]:
# ── FedProx MLP ──
class CheckpointFedProx(fl.server.strategy.FedProx):
    def __init__(self, strategy_name, model_dir_path, **kw):
        super().__init__(**kw)
        self.strategy_name = strategy_name
        self.model_dir_path = Path(model_dir_path) / strategy_name
        self.model_dir_path.mkdir(parents=True, exist_ok=True)
    def aggregate_fit(self, server_round, results, failures):
        round_dir = self.model_dir_path / f"round_{server_round:03d}"; round_dir.mkdir(parents=True, exist_ok=True)
        for cp, fit_res in results:
            ndarrays = fl.common.parameters_to_ndarrays(fit_res.parameters)
            m = build_mlp(BEST_MLP_DH); numpy_to_model(m, ndarrays)
            torch.save(m.state_dict(), round_dir / f"client_{cp.cid}.pt")
        aggregated, metrics = super().aggregate_fit(server_round, results, failures)
        if aggregated is not None:
            nd = fl.common.parameters_to_ndarrays(aggregated)
            gm = build_mlp(BEST_MLP_DH); numpy_to_model(gm, nd)
            torch.save(gm.state_dict(), round_dir / "global_model.pt")
        return aggregated, metrics

fedprox_histories = {}
for mu in FEDPROX_MUS:
    print(f"\n=== FedProx MLP (mu={mu}, {best_strategy} mask) ===")
    fedprox_hist = []; _eval_fn = get_mlp_eval_fn(client_test_pp, BEST_MLP_THRESH, fedprox_hist)
    s = CheckpointFedProx(f"fedprox_mlp_mu{mu}_{best_strategy}", MODEL_DIR,
        fraction_fit=1.0, fraction_evaluate=0.0,
        min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4,
        proximal_mu=mu, evaluate_fn=_eval_fn,
        initial_parameters=fl.common.ndarrays_to_parameters(model_to_numpy(build_mlp(BEST_MLP_DH))))
    s.model_dir_path = Path(MODEL_DIR) / f"fedprox_mlp_mu{mu}_{best_strategy}"

    def make_client_fn(train_dict):
        def _fn(cid):
            site = SITE_ORDER[int(cid)]
            return FedMLPClient(cid, *train_dict[site]).to_client()
        return _fn

    fl.simulation.start_simulation(
        client_fn=make_client_fn(client_train_pp), num_clients=4,
        config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
        strategy=s, client_resources={"num_cpus": 1, "num_gpus": 0})
    fedprox_histories[mu] = fedprox_hist.copy()
    print(f"FedProx mu={mu} done. {len(fedprox_hist)} rounds.")

In [ ]:
# ── FedLR with tuned C ──
print("\n=== FedAvg LR (tuned C) ===")
C_GRID = np.linspace(5e-5, 1e-3, 15)
site_lr_scores = {}
for site in SITE_ORDER:
    X_tr, y_tr = client_train_pp[site]
    grid = GridSearchCV(LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced",
                                             max_iter=5000, random_state=SEED),
                        param_grid={"C": C_GRID}, cv=3, scoring="balanced_accuracy", n_jobs=-1)
    grid.fit(X_tr, y_tr)
    for i, params in enumerate(grid.cv_results_["params"]):
        key = float(params["C"])
        score = grid.cv_results_["mean_test_score"][i]
        site_lr_scores.setdefault(key, []).append(score)
BEST_LR_C = max(site_lr_scores, key=lambda c: min(site_lr_scores[c]))
print(f"Tuned LR C (worst-site): {BEST_LR_C:.2e}")

# Per-site LR threshold tuning (worst-site optimal)
lr_thresh_scores = {t: [] for t in THRESHOLDS}
for site in SITE_ORDER:
    X_tr, y_tr = client_train_pp[site]
    cv_proba = cross_val_predict(
        LogisticRegression(C=BEST_LR_C, penalty="l2", solver="lbfgs",
                           class_weight="balanced", max_iter=5000, random_state=SEED),
        X_tr, y_tr, cv=3, method="predict_proba", n_jobs=-1)[:, 1]
    for t in THRESHOLDS:
        lr_thresh_scores[t].append(balanced_accuracy_score(y_tr, cv_proba >= t))
BEST_LR_THRESH = max(THRESHOLDS, key=lambda t: min(lr_thresh_scores[t]))
print(f"Tuned LR threshold (worst-site): {BEST_LR_THRESH:.3f}")

class FedLRClient(fl.client.NumPyClient):
    def __init__(self, cid, X_train, y_train):
        self.cid = cid; self.X_train, self.y_train = X_train, y_train
        nf = X_train.shape[1]
        self.model = LogisticRegression(C=BEST_LR_C, penalty="l2", solver="saga", max_iter=1,
                                        warm_start=True, class_weight="balanced", random_state=SEED)
        self.model.classes_ = np.array([0,1]); self.model.coef_ = np.zeros((1,nf)); self.model.intercept_ = np.zeros(1)
    def get_parameters(self, config): return [self.model.coef_.ravel(), self.model.intercept_]
    def set_parameters(self, params):
        self.model.coef_ = params[0].reshape(1,-1); self.model.intercept_ = params[1]
    def fit(self, parameters, config):
        self.set_parameters(parameters)
        with warnings.catch_warnings(): warnings.simplefilter("ignore"); self.model.fit(self.X_train, self.y_train)
        return (self.get_parameters({}), len(self.X_train), {"num_examples": len(self.X_train)})

def get_lr_eval_fn(test_dict, hist_list):
    def evaluate(server_round, parameters, config):
        lr = LogisticRegression(C=BEST_LR_C, penalty="l2", solver="lbfgs", class_weight="balanced")
        lr.classes_ = np.array([0,1]); lr.coef_ = parameters[0].reshape(1,-1); lr.intercept_ = parameters[1]
        record = {"round": server_round}; ap, al = [], []
        for site in SITE_ORDER:
            X_tt, y_tt = test_dict[site]
            proba = lr.predict_proba(X_tt)[:,1]
            preds = proba >= BEST_LR_THRESH
            record[f"{site}_BalAcc"] = float(balanced_accuracy_score(y_tt, preds))
            record[f"{site}_AUC"] = float(roc_auc_score(y_tt, proba))
            ap.append(proba); al.append(y_tt)
        apc = np.concatenate(ap); alc = np.concatenate(al)
        record["All_BalAcc"] = float(balanced_accuracy_score(alc, apc >= BEST_LR_THRESH))
        record["All_AUC"] = float(roc_auc_score(alc, apc))
        hist_list.append(record)
        return (1.0 - record["All_BalAcc"], record)
    return evaluate

fedlr_hist = []; _eval_fn = get_lr_eval_fn(client_test_pp, fedlr_hist)
n_feat = client_train_pp[SITE_ORDER[0]][0].shape[1]
lr_init = LogisticRegression(C=BEST_LR_C, penalty="l2", solver="saga", max_iter=1,
                              warm_start=True, class_weight="balanced", random_state=SEED)
lr_init.classes_ = np.array([0,1]); lr_init.coef_ = np.zeros((1,n_feat)); lr_init.intercept_ = np.zeros(1)
lr_strat = fl.server.strategy.FedAvg(
    fraction_fit=1.0, fraction_evaluate=0.0,
    min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4, evaluate_fn=_eval_fn,
    initial_parameters=fl.common.ndarrays_to_parameters([lr_init.coef_.ravel(), lr_init.intercept_]))

def lr_client_fn(cid):
    site = SITE_ORDER[int(cid)]
    return FedLRClient(cid, *client_train_pp[site]).to_client()

fl.simulation.start_simulation(client_fn=lr_client_fn, num_clients=4,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=lr_strat, client_resources={"num_cpus": 1, "num_gpus": 0})
print(f"FedLR done. {len(fedlr_hist)} rounds.")

In [ ]:
# ── FedRF tree collection ──
def trees_to_array(est):
    buf = io.BytesIO(); joblib.dump(est, buf); buf.seek(0)
    return np.frombuffer(buf.read(), dtype=np.uint8)
def array_to_trees(arr):
    return joblib.load(io.BytesIO(arr.tobytes()))

class TreeCollectionFedAvg(fl.server.strategy.FedAvg):
    def __init__(self, **kw):
        super().__init__(**kw); self._accumulated_trees = []
    def aggregate_fit(self, server_round, results, failures):
        if not results: return None, {}
        for _, fit_res in results:
            ndarrays = fl.common.parameters_to_ndarrays(fit_res.parameters)
            if len(ndarrays[0]) > 0: self._accumulated_trees.extend(array_to_trees(ndarrays[0]))
        combined = trees_to_array(self._accumulated_trees)
        aggregated = fl.common.ndarrays_to_parameters([combined])
        metrics = {"total_trees": len(self._accumulated_trees)}
        return aggregated, metrics

class FedRFClient(fl.client.NumPyClient):
    def __init__(self, cid, X_train, y_train):
        self.cid = cid; self.X_train, self.y_train = X_train, y_train
    def get_parameters(self, config): return [np.array([], dtype=np.uint8)]
    def fit(self, parameters, config):
        server_round = int(config.get("current_round", 0))
        rf = RandomForestClassifier(**RF_PARAMS, random_state=SEED+int(self.cid)+100*server_round,
                                     n_jobs=-1, warm_start=True)
        rf.fit(self.X_train, self.y_train)
        return ([trees_to_array(list(rf.estimators_))], len(self.X_train),
                {"n_new_trees": len(rf.estimators_), "num_examples": len(self.X_train)})

def get_rf_eval_fn(test_dict, hist_list):
    def evaluate(server_round, parameters, config):
        if len(parameters[0]) == 0: return (1.0, {"All_BalAcc": 0.5})
        trees = array_to_trees(parameters[0])
        rf = RandomForestClassifier(**RF_PARAMS, n_jobs=-1); rf.estimators_ = trees
        rf.n_classes_ = 2; rf.classes_ = np.array([0,1]); rf.n_outputs_ = 1
        record = {"round": server_round, "n_trees": len(trees)}; ap, al = [], []
        for site in SITE_ORDER:
            X_tt, y_tt = client_test_pp[site]
            proba = rf.predict_proba(X_tt)[:,1]
            preds = proba >= BEST_RF_THRESH
            record[f"{site}_BalAcc"] = float(balanced_accuracy_score(y_tt, preds))
            record[f"{site}_AUC"] = float(roc_auc_score(y_tt, proba))
            ap.append(proba); al.append(y_tt)
        apc = np.concatenate(ap); alc = np.concatenate(al)
        record["All_BalAcc"] = float(balanced_accuracy_score(alc, apc >= BEST_RF_THRESH))
        record["All_AUC"] = float(roc_auc_score(alc, apc))
        hist_list.append(record)
        return (1.0 - record["All_BalAcc"], record)
    return evaluate

print("\n=== FedRF (Tree Collection) ===")
fedrf_hist = []; _eval_fn = get_rf_eval_fn(client_test_pp, fedrf_hist)
rf_strat = TreeCollectionFedAvg(
    fraction_fit=1.0, fraction_evaluate=0.0,
    min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4, evaluate_fn=_eval_fn,
    initial_parameters=fl.common.ndarrays_to_parameters([np.array([], dtype=np.uint8)]))

def rf_client_fn(cid):
    site = SITE_ORDER[int(cid)]
    return FedRFClient(cid, *client_train_pp[site]).to_client()

fl.simulation.start_simulation(client_fn=rf_client_fn, num_clients=4,
    config=fl.server.ServerConfig(num_rounds=NUM_RF_ROUNDS),
    strategy=rf_strat, client_resources={"num_cpus": 1, "num_gpus": 0})
fedrf_hist = fedrf_hist.copy()  # noqa F841
print(f"FedRF done. {len(fedrf_hist)} rounds.")

In [ ]:
# ── Cross-Site MLP + RF (best mask, persite-aware) ──
print("\n=== Cross-Site MLP + RF (A -> B/C/D) ===")
X_A_train, y_A_train = client_train["A"]
state_cs = fit_input_transform(X_A_train, "log1p+standardize")
X_A_tr = apply_input_transform(X_A_train, state_cs)

# Apply best mask (persite-aware)
mask_A = np.array([], dtype=int)
if best_strategy == "union":
    mask_A = union_mask
elif best_strategy == "majority":
    mask_A = majority_mask
elif best_strategy == "persite":
    mask_A = site_masks_raw["A"] if len(site_masks_raw["A"]) > 0 else np.array([], dtype=int)
if len(mask_A) > 0:
    X_A_tr[:, mask_A] = 0.0

cross_test_sets = {}
for sk in "BCD":
    X_te, y_te = client_test[sk]
    X_te_pp = apply_input_transform(X_te, state_cs)
    # Per-site mask
    if best_strategy == "union":
        X_te_pp[:, union_mask] = 0.0
    elif best_strategy == "majority":
        X_te_pp[:, majority_mask] = 0.0
    elif best_strategy == "persite":
        mask_site = site_masks_raw[sk] if len(site_masks_raw[sk]) > 0 else np.array([], dtype=int)
        if len(mask_site) > 0: X_te_pp[:, mask_site] = 0.0
    cross_test_sets[sk] = (X_te_pp, y_te)
# A-val
X_te_pp = apply_input_transform(client_test["A"][0], state_cs)
if len(mask_A) > 0: X_te_pp[:, mask_A] = 0.0
cross_test_sets["A"] = (X_te_pp, client_test["A"][1])

# ── Cross-site MLP ──
cs_m = build_mlp(BEST_MLP_DH).to(DEV)
ds_cs = DatasetFromNumpy(X_A_tr, y_A_train)
dl_cs = DataLoader(ds_cs, batch_size=64, shuffle=True)
opt_cs = torch.optim.AdamW(cs_m.parameters(), lr=BEST_MLP_LR, weight_decay=1e-4)
sched_cs = torch.optim.lr_scheduler.CosineAnnealingLR(opt_cs, T_max=90, eta_min=1e-6)
crit_cs = nn.CrossEntropyLoss()
for ep in range(100):
    cs_m.train()
    for xb, yb in dl_cs:
        if len(xb) <= 1: continue
        xb, yb = xb.to(DEV), yb.to(DEV)
        if ep < 10:
            for pg in opt_cs.param_groups: pg["lr"] = BEST_MLP_LR * (ep+1)/10
        opt_cs.zero_grad(); crit_cs(cs_m(xb), yb).backward(); opt_cs.step()
    if ep >= 10: sched_cs.step()
cs_m.eval()
cross_site_results = {}
for name, (X_tt, y_tt) in cross_test_sets.items():
    proba = mlp_predict_proba(cs_m, X_tt)
    cross_site_results[f"{name}_BalAcc"] = balanced_accuracy_score(y_tt, proba >= BEST_MLP_THRESH)
    cross_site_results[f"{name}_AUC"] = roc_auc_score(y_tt, proba)
    print(f"  CS MLP {name}: BalAcc={cross_site_results[f'{name}_BalAcc']:.4f}  AUC={cross_site_results[f'{name}_AUC']:.4f}")
X_BCD = np.concatenate([cross_test_sets[s][0] for s in "BCD"])
y_BCD = np.concatenate([cross_test_sets[s][1] for s in "BCD"])
proba_bcd = mlp_predict_proba(cs_m, X_BCD)
cross_site_results["All_BalAcc"] = balanced_accuracy_score(y_BCD, proba_bcd >= BEST_MLP_THRESH)
cross_site_results["All_AUC"] = roc_auc_score(y_BCD, proba_bcd)
print(f"  CS MLP All (B+C+D): BalAcc={cross_site_results['All_BalAcc']:.4f}  AUC={cross_site_results['All_AUC']:.4f}")

# ── Cross-site RF ──
print("\n=== Cross-Site RF (A -> B/C/D) ===")
rf_cs = RandomForestClassifier(**RF_PARAMS, random_state=SEED, n_jobs=-1)
rf_cs.fit(X_A_tr, y_A_train)
cross_site_rf_results = {}
for name, (X_tt, y_tt) in cross_test_sets.items():
    proba = rf_cs.predict_proba(X_tt)[:, 1]
    cross_site_rf_results[f"{name}_BalAcc"] = balanced_accuracy_score(y_tt, proba >= BEST_RF_THRESH)
    cross_site_rf_results[f"{name}_AUC"] = roc_auc_score(y_tt, proba)
    print(f"  CS RF  {name}: BalAcc={cross_site_rf_results[f'{name}_BalAcc']:.4f}  AUC={cross_site_rf_results[f'{name}_AUC']:.4f}")
proba_bcd_rf = rf_cs.predict_proba(X_BCD)[:, 1]
cross_site_rf_results["All_BalAcc"] = balanced_accuracy_score(y_BCD, proba_bcd_rf >= BEST_RF_THRESH)
cross_site_rf_results["All_AUC"] = roc_auc_score(y_BCD, proba_bcd_rf)
print(f"  CS RF  All (B+C+D): BalAcc={cross_site_rf_results['All_BalAcc']:.4f}  AUC={cross_site_rf_results['All_AUC']:.4f}")

In [ ]:
# ── Assemble final results DataFrame ──
def best_metrics(h):
    if not h: return {}, 0
    skip0 = h[1:]
    best = max(skip0, key=lambda r: r.get("All_BalAcc", 0))
    return best, int(best.get("round", 0))

rows = []
def add_row(method, cs_source=None, fed_hist=None):
    r = {"Method": method}; peak_round = ""
    for site in SITE_ORDER:
        if cs_source:
            r[f"{site}_BalAcc"] = cs_source.get(f"{site}_BalAcc", np.nan)
            r[f"{site}_AUC"] = cs_source.get(f"{site}_AUC", np.nan)
        elif fed_hist is not None:
            lm, pr = best_metrics(fed_hist)
            r[f"{site}_BalAcc"] = lm.get(f"{site}_BalAcc", np.nan)
            r[f"{site}_AUC"] = lm.get(f"{site}_AUC", np.nan)
            peak_round = f" (r{pr})"
    if fed_hist is not None:
        lm, pr = best_metrics(fed_hist)
        r["All_BalAcc"] = lm.get("All_BalAcc", np.nan); r["All_AUC"] = lm.get("All_AUC", np.nan)
        r["Peak_Round"] = int(pr)
    elif cs_source:
        r["All_BalAcc"] = cs_source.get("All_BalAcc", np.nan); r["All_AUC"] = cs_source.get("All_AUC", np.nan)
        r["Peak_Round"] = 0
    r["Label"] = method + peak_round; rows.append(r)

add_row("Centralized MLP (unmasked)", cs_source=centralized_mlp_results)
add_row("Centralized RF (unmasked)", cs_source=centralized_rf_results)
for strategy in MASK_STRATEGIES:
    add_row(f"Centralized MLP ({strategy})", cs_source=mask_comparison[strategy])
if "fedavg_hist" in dir() and fedavg_hist:
    add_row(f"FL FedAvg MLP ({best_strategy})", fed_hist=fedavg_hist)
for mu in FEDPROX_MUS:
    if mu in fedprox_histories and fedprox_histories[mu]:
        add_row(f"FL FedProx mu={mu} ({best_strategy})", fed_hist=fedprox_histories[mu])
if "fedlr_hist" in dir(): add_row("FL FedAvg LR", fed_hist=fedlr_hist)
if "fedrf_hist" in dir(): add_row("FL FedRF (Trees)", fed_hist=fedrf_hist)
add_row("Cross-Site MLP", cs_source=cross_site_results)
add_row("Cross-Site RF", cs_source=cross_site_rf_results)

df_results = pd.DataFrame(rows)
cols = ["Method", "Label", "Peak_Round"] + [f"{s}_BalAcc" for s in SITE_ORDER] + ["All_BalAcc"]
print(df_results[cols].to_string(index=False))

In [ ]:
# ── Convergence plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
for strategy in MASK_STRATEGIES:
    h = fedavg_hist_per_mask[strategy]
    vals = [r.get("All_BalAcc", np.nan) for r in h]
    ax.plot(range(1, len(vals)+1), vals, label=f"{strategy} mask")
ax.set_title(f"FedAvg MLP — Mask Comparison"); ax.set_xlabel("Round"); ax.set_ylabel("All_BalAcc")
ax.legend(fontsize=8); ax.grid(True, ls='--', alpha=0.5); ax.set_ylim(0.3, 1.0)

ax = axes[1]
plots = [("FedAvg MLP", fedavg_hist, "#ff7f0e", "-")]
for mu in FEDPROX_MUS:
    if mu in fedprox_histories: plots.append((f"FedProx mu={mu}", fedprox_histories[mu], "#d62728", "--"))
plots.append(("FedAvg LR", fedlr_hist, "#1f77b4", "-."))
plots.append(("FedRF", fedrf_hist, "#2ca02c", "-"))
for label, hist, c, ls in plots:
    vals = [h.get("All_BalAcc", np.nan) for h in hist]
    ax.plot(range(1, len(vals)+1), vals, color=c, ls=ls, lw=2, label=label)
ax.axhline(centralized_mlp_results["All_BalAcc"], color='gray', ls=':', lw=2, label='Centralized MLP (unmasked)')
ax.set_title(f"All Methods ({best_strategy} mask)"); ax.set_xlabel("Round"); ax.set_ylabel("All_BalAcc")
ax.legend(fontsize=7); ax.grid(True, ls='--', alpha=0.5); ax.set_ylim(0.3, 1.0)

fig.suptitle(f"{DRUG_NAME} — Species-Masked Federated Learning", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT_DIR / "convergence.pdf", bbox_inches="tight"); plt.show()

In [ ]:
ba_data = {}
for _, r in df_results.iterrows():
    ba_data[r["Label"]] = {f"Site {s}": r[f"{s}_BalAcc"] for s in SITE_ORDER}
    if not np.isnan(r.get("All_BalAcc", np.nan)): ba_data[r["Label"]]["All"] = r["All_BalAcc"]
df_ba = pd.DataFrame(ba_data).T
df_ba = df_ba[[c for c in [f"Site {s}" for s in SITE_ORDER]+["All"] if c in df_ba.columns]]
fig, ax = plt.subplots(figsize=(10, max(4, len(df_ba)*0.5)))
sns.heatmap(df_ba, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.4, vmax=1.0,
            linewidths=1.0, linecolor="white", cbar_kws={"label":"Balanced Accuracy"}, ax=ax)
ax.set_title(f"{DRUG_NAME} — Species-Masked BalAcc ({best_strategy})", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT_DIR / "heatmap_balacc.pdf", bbox_inches="tight"); plt.show()

auc_data = {}
for _, r in df_results.iterrows():
    auc_data[r["Label"]] = {f"Site {s}": r[f"{s}_AUC"] for s in SITE_ORDER}
    if not np.isnan(r.get("All_AUC", np.nan)): auc_data[r["Label"]]["All"] = r["All_AUC"]
df_auc = pd.DataFrame(auc_data).T
df_auc = df_auc[[c for c in [f"Site {s}" for s in SITE_ORDER]+["All"] if c in df_auc.columns]]
fig, ax = plt.subplots(figsize=(10, max(4, len(df_auc)*0.5)))
sns.heatmap(df_auc, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.4, vmax=1.0,
            linewidths=1.0, linecolor="white", cbar_kws={"label":"AUC-ROC"}, ax=ax)
ax.set_title(f"{DRUG_NAME} — Species-Masked AUC ({best_strategy})", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT_DIR / "heatmap_auc.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# ── Mask Δ vs unmasked ──
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(SITE_ORDER)); w = 0.2
for i, strategy in enumerate(["union", "majority", "persite"]):
    deltas = [mask_comparison[strategy].get(f"{site}_BalAcc", 0) - mask_comparison["none"].get(f"{site}_BalAcc", 0)
              for site in SITE_ORDER]
    ax.bar(x + (i-1)*w, deltas, w, label=strategy)
ax.set_xticks(x); ax.set_xticklabels([f"Site {s}" for s in SITE_ORDER])
ax.axhline(0, color='black', lw=1)
ax.set_ylabel("Δ BalAcc (masked - unmasked)"); ax.set_title("Species Masking Effect per Site (Centralized MLP)")
ax.legend(); ax.grid(True, ls='--', alpha=0.5, axis='y')
plt.tight_layout(); plt.savefig(OUT_DIR / "mask_delta.pdf", bbox_inches="tight"); plt.show()

In [ ]:
df_results.to_csv(OUT_DIR / "final_results.csv", index=False)
pd.DataFrame(fedavg_hist).to_csv(OUT_DIR / "fedavg_per_round.csv", index=False)
pd.DataFrame(fedlr_hist).to_csv(OUT_DIR / "fedlr_per_round.csv", index=False)
pd.DataFrame(fedrf_hist).to_csv(OUT_DIR / "fedrf_per_round.csv", index=False)
for mu in FEDPROX_MUS:
    if mu in fedprox_histories:
        pd.DataFrame(fedprox_histories[mu]).to_csv(OUT_DIR / f"fedprox_mu{mu}_per_round.csv", index=False)

# Save best params
with open(OUT_DIR / "best_params_used.txt", "w") as f:
    f.write(f"DRUG={DRUG_NAME}\n")
    f.write(f"BEST_MASK={best_strategy}\n")
    f.write(f"BEST_MLP_LR={BEST_MLP_LR}\n")
    f.write(f"BEST_MLP_DH={BEST_MLP_DH}\n")
    f.write(f"BEST_MLP_THRESH={BEST_MLP_THRESH}\n")
    f.write(f"BEST_LR_C={BEST_LR_C}\n")
    f.write(f"BEST_LR_THRESH={BEST_LR_THRESH}\n")
    f.write(f"RF_PARAMS={RF_PARAMS}\n")
    f.write(f"BEST_RF_THRESH={BEST_RF_THRESH}\n")

# Archive
src_nb = Path("06-03c-Species-Masked-Ceftriaxone-Federated.ipynb")
if not src_nb.exists():
    import glob as _g
    candidates = list(_g.glob("/content/**/06-03c*.ipynb", recursive=True))
    if candidates: src_nb = Path(candidates[0])
if src_nb.exists():
    shutil.copy(str(src_nb), str(OUT_DIR / "notebook.ipynb"))
    print("Notebook archived.")

print(f"\n{'='*60}")
print(f"  Done. Results in {OUT_DIR.resolve()}")
print(f"  Best mask: {best_strategy}")
for f in sorted(OUT_DIR.glob("*")): print(f"    {f.name}")

---
**Done.** Species-masked federated analysis complete.

| Output | Location |
|---|---|
| RF species models | `rf_species_models/site_{A,B,C,D}_rf_species.joblib` |
| Masks | `masks/{union,majority,persite}/` |
| Results | `results/` |
| Models | `models/{strategy}/` |